# Self-Healing Agentic RAG — reproduce the results

This notebook runs the project end to end on a free Colab CPU/GPU runtime:

1. install + run the offline test suite
2. a 5-question demo of the self-healing loop
3. **retrieval benchmark** (BM25 vs dense vs hybrid) — *no API key needed*
4. **hallucination-detector benchmark** on RAGBench — heuristic + NLI tiers need *no API key*
5. **end-to-end QA** on HotpotQA: vanilla RAG vs self-healing (needs a free Groq or Gemini key)

Every evaluation writes `results/<tag>/report.md`; copy those tables into the README.
Runs resume where they stopped, so free-tier rate limits are not a problem — just re-run the cell.

In [ ]:
REPO_URL = "https://github.com/kwazzy-coder/Self-Healing-Agentic-RAG-System"
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

### API key
Add a key in Colab's **🔑 Secrets** panel (left sidebar) named `GROQ_API_KEY`
(free at https://console.groq.com/keys) or `GEMINI_API_KEY` (free at https://aistudio.google.com/apikey).
Sections 1–4 work without any key.

In [ ]:
import os
try:
    from google.colab import userdata
    for name in ("GROQ_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass
os.environ["LLM_BACKEND"] = "groq" if os.environ.get("GROQ_API_KEY") else "gemini"
print("LLM backend:", os.environ["LLM_BACKEND"], "| key set:", any(os.environ.get(k) for k in ("GROQ_API_KEY", "GEMINI_API_KEY")))

## 1. Tests (offline, ~5 s)

In [ ]:
!python -m pytest -q

## 2. Demo

In [ ]:
!python demo.py

## 3. Retrieval benchmark (no API key)

In [ ]:
!python -m evaluation.benchmark_retrieval --n_samples 500

## 4. Hallucination-detector benchmark on RAGBench

The monitor judges RAGBench's *given* responses; we compare with the human/GPT-4 grounding labels.
`heuristic` and `heuristic_nli` are free; `tiered` and `llm` call the LLM judge.

In [ ]:
!python run_evaluation.py --dataset ragbench --subset covidqa --mode detector --n_samples 200 \
    --monitors heuristic,heuristic_nli

In [ ]:
# Needs an API key: adds the LLM-judge tier (tiered) and the always-judge baseline (llm)
!python run_evaluation.py --dataset ragbench --subset covidqa --mode detector --n_samples 200 \
    --monitors heuristic,heuristic_nli,tiered,llm

## 5. End-to-end QA on HotpotQA (needs an API key)

Start with 50 questions; the run resumes if interrupted. `--systems` can include
`vanilla_rag, planner_only, self_healing, nli_only, llm_judge_only, always_correct, crag`.

In [ ]:
!python run_evaluation.py --dataset hotpotqa --n_samples 50 \
    --systems vanilla_rag,planner_only,self_healing,llm_judge_only

## 6. Look at the results

In [ ]:
import glob
from IPython.display import Markdown, display
for path in sorted(glob.glob("results/*/report.md")):
    display(Markdown(open(path).read()))

In [ ]:
# Where does it fail, and what did healing fix / break?
!python -m analysis.error_analysis results/hotpotqa_qa_n50_s42/self_healing.jsonl --cases 3

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")